# Exploratoty Data Analysis(EDA)

### 1. Import Libraries

In [ ]:
import pandas as pd

### 2. Load dataset & view dataset

We can detect that the names of our columns contains space characters and capitalization of the first letters, these are not standard naming conventions for columns and can cause issues latter, so we will be renaming the columns by replacing the space columns with underscores and using lower cases.

In [ ]:
df = pd.read_csv("customer_shopping_behavior.csv")
df.head(3)

# Documents/Data_Science_Jupyter/My articles/github_files/customer_behavior_analysis/customer_shopping_behavior.csv

In [ ]:
df.columns = df.columns.str.replace(" ", "_")
df.head(3)

In [ ]:
df.columns = df.columns.str.lower()
df.head(3)

In [ ]:
df = df.rename(columns={"purchase_amount_(usd)": "purchase_amount_usd"})
df.head(3)

### 3. Explore the dataset

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
df.isnull().sum()

There are 37 null values in the review_ratings, this will affect our computations later. We have the options to fill with the mean, median or NaN values. For our use case, median fits best since it has the least pronounced effects on our analysis later.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.hist(df["review_rating"])

In [ ]:
df['review_rating'] = df.groupby("category")['review_rating'].transform(lambda x: x.fillna(x.median()))
df.isnull().sum()

## Questions we plan on answering
1. What is the total revenue generated by male vs. female customers?
2. Which customers used a discount but still spent more than the average purchase amount? 
3. Which are the top 5 products with the highest average review rating?
4. Compare the average Purchase Amounts between Standard and Express Shipping. 
5. Do subscribed customers spend more? Compare average spend and total revenue between subscribers and non-subscribers.
6. Which 5 products have the highest percentage of purchases with discounts applied?
7. Segment customers into New, Returning, and Loyal based on their total 
8. What are the top 3 most purchased products within each category? 
9. Are customers who are repeat buyers (more than 5 previous purchases) also likely to subscribe?
10. What is the revenue contribution of each age group? 

Some of the above questions cannot be answered without some new features(columns) to be added to our dataset. Features like:
- Age group
- Purchase frequency in days

### 4. Feature Engineering

In [ ]:
# The required age groups are: 'Young Adult', 'Adult', 'Middle Age', 'Elder'
labels = ['Young Adult', 'Adult', 'Middle Age', 'Elder']

df['age_group'] = pd.qcut(df['age'], q=4, labels = labels)
df[['age', 'age_group']]

In [ ]:
df['frequency_of_purchases'].unique()

In [ ]:
labels = {
    'Fortnightly': 14,
    'Weekly': 7, 
    'Annually': 365, 
    'Quarterly': 90, 
    'Bi-Weekly': 14,
    'Monthly': 30,
    'Every 3 Months': 90
}

df['frequency_of_days'] = df['frequency_of_purchases'].map(labels)

In [ ]:
df[['frequency_of_purchases', 'frequency_of_days']]

In [ ]:
(df['discount_applied'] == df['promo_code_used']).all()

In [ ]:
df = df.drop('promo_code_used', axis=1)

In [ ]:
df.head(5)

## 5. Connecting our dataset to MySQL database

In [ ]:
# pip install psycopg2-binary sqlalchemy pymysql dotenv

In [ ]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

# MySQL connection
username = os.getenv("DB_USER")
password = os.getenv("DB_PASS")
host = "localhost"
port = "3306"
database = "customer_behavior_analysis"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

# Write DataFrame to MySQL
table_name = "customer"   # choose any table name
df.to_sql(table_name, engine, if_exists="replace", index=False)

# Read back sample
pd.read_sql("SELECT * FROM customer LIMIT 5;", engine)